# Geometry Dataset Balance Check

This notebook visualizes label distribution from `dataset/data/diagrams_filter.json`
and reports simple class-balance metrics.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
DATA_PATH = Path("../dataset/data/diagrams_filter.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Cannot find file: {DATA_PATH.resolve()}")

df = pd.read_json(DATA_PATH)
df = df.fillna({"caption": "", "caption_vn": ""})
df.head()

In [ ]:
def extract_primary_category(caption: str) -> str:
    # Use the first phrase before comma as a simple class label proxy.
    text = str(caption).strip().lower()
    if not text:
        return "unknown"
    return text.split(",", 1)[0].strip()

KEYWORDS = [
    "triangle", "circle", "rectangle", "square",
    "incircle", "circumcircle", "diameter", "midpoint",
    "perpendicular", "parallel", "bisector",
    "orthocenter", "incenter", "centroid", "tangent"
]

df["primary_category"] = df["caption"].map(extract_primary_category)
df["num_clauses"] = df["caption"].map(lambda x: str(x).count(",") + 1 if str(x).strip() else 0)

for kw in KEYWORDS:
    df[f"kw_{kw}"] = df["caption"].str.lower().str.contains(kw, regex=False).astype(int)

df[["id", "caption", "primary_category", "num_clauses"]].head()

In [ ]:
category_counts = df["primary_category"].value_counts()
top_categories = category_counts.head(15).sort_values()

keyword_counts = {kw: int(df[f"kw_{kw}"].sum()) for kw in KEYWORDS}
keyword_counts = dict(sorted(keyword_counts.items(), key=lambda x: x[1], reverse=True))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Geometry Dataset Balance Overview", fontsize=16, fontweight="bold")

# 1) Top primary categories
axes[0, 0].barh(top_categories.index, top_categories.values)
axes[0, 0].set_title("Top 15 Primary Categories")
axes[0, 0].set_xlabel("Count")

# 2) Clause complexity
axes[0, 1].hist(df["num_clauses"], bins=20, edgecolor="black")
axes[0, 1].set_title("Caption Complexity (Number of Clauses)")
axes[0, 1].set_xlabel("Clauses per sample")
axes[0, 1].set_ylabel("Frequency")

# 3) Keyword counts
axes[1, 0].bar(keyword_counts.keys(), keyword_counts.values())
axes[1, 0].set_title("Keyword Frequency")
axes[1, 0].set_ylabel("Count")
axes[1, 0].tick_params(axis="x", rotation=45)

# 4) Dominance view (top 8 + others)
top8 = category_counts.head(8)
others = int(category_counts.iloc[8:].sum())
labels = list(top8.index) + (["others"] if others > 0 else [])
sizes = list(top8.values) + ([others] if others > 0 else [])
axes[1, 1].pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90)
axes[1, 1].set_title("Top Categories Share")

plt.tight_layout()
plt.show()

In [ ]:
def normalized_entropy(counts: np.ndarray) -> float:
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    if len(probs) <= 1:
        return 0.0
    h = -np.sum(probs * np.log(probs))
    return float(h / np.log(len(probs)))

counts = category_counts.values.astype(float)
max_count = counts.max() if len(counts) else 0
min_count = counts.min() if len(counts) else 0
imbalance_ratio = (max_count / min_count) if min_count > 0 else np.inf
entropy_score = normalized_entropy(counts) if len(counts) else 0.0

print(f"Total samples: {len(df)}")
print(f"Number of primary categories: {category_counts.size}")
print(f"Largest class size: {int(max_count)}")
print(f"Smallest class size: {int(min_count)}")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
print(f"Normalized entropy [0,1]: {entropy_score:.3f}")

if imbalance_ratio <= 2 and entropy_score >= 0.90:
    verdict = "Relatively balanced"
elif imbalance_ratio <= 5 and entropy_score >= 0.75:
    verdict = "Moderately imbalanced"
else:
    verdict = "Strongly imbalanced"

print(f"Balance verdict: {verdict}")

print("\nTop 20 categories:")
display(category_counts.head(20).to_frame("count"))

## Notes
- This notebook uses `primary_category` extracted from the first phrase in `caption`.
- If you have official labels, replace `extract_primary_category` with your true class field for a stricter balance check.